# 🧪 Semana 15 · Unidad 4 — Laboratorio: Árboles balanceados

**Universidad de Talca — Curso de Algoritmos y Estructuras de Datos**

| Aspecto | Detalle |
|--------|--------|
| **Profesor** | PhD. César Astudillo |
| **Unidad** | Unidad 4: Diccionarios |
| **Tema** | Árboles rojo-negro: rotaciones, invariantes y garantía de altura |
| **Duración** | 100 minutos (2 bloques de 50 min) |

---

## 📋 Instrucciones Generales

- Trabaja de forma **individual**.
- Ejecuta cada celda antes de pasar a la siguiente.
- El verificador automático te dirá cuántos casos pasan. Apunta a 100%.
- Al terminar el Bloque 1, el ayudante revisará tu avance antes de continuar.
- **No modifiques** las celdas de Setup ni las de verificación.

> 📌 **Prerrequisito.** Este laboratorio continúa el BST de la Semana 14 y la cátedra de
> hoy sobre árboles 2-3 y rojo-negro.

> ⚠️ **Entrega de la Tarea de la Unidad 4** al final de esta sesión.

In [ ]:
# Setup — ejecutar primero. No modificar.
import sys, math, random, time
import numpy as np
import matplotlib.pyplot as plt

random.seed(2026)
np.random.seed(2026)

ROJO, NEGRO = True, False


class Nodo:
    """Nodo de un árbol rojo-negro left-leaning. El color es el del enlace de entrada."""
    __slots__ = ("clave", "valor", "izq", "der", "color", "n")

    def __init__(self, clave, valor, color=ROJO):
        self.clave, self.valor = clave, valor
        self.izq = self.der = None
        self.color = color
        self.n = 1


def es_rojo(nodo) -> bool:
    """True si el enlace que entra a `nodo` es rojo. Un enlace nulo es NEGRO."""
    return nodo is not None and nodo.color == ROJO


def tam(nodo) -> int:
    """Tamaño del subárbol; 0 si es nulo."""
    return 0 if nodo is None else nodo.n


class NodoBST:
    """Nodo de un BST simple, para la comparación del Bloque 2."""
    __slots__ = ("clave", "valor", "izq", "der")

    def __init__(self, clave, valor):
        self.clave, self.valor = clave, valor
        self.izq = self.der = None


def bst_insertar(raiz, clave, valor):
    """Inserta en un BST simple (iterativo, para soportar árboles degenerados)."""
    if raiz is None:
        return NodoBST(clave, valor)
    x = raiz
    while True:
        if clave < x.clave:
            if x.izq is None:
                x.izq = NodoBST(clave, valor); return raiz
            x = x.izq
        elif clave > x.clave:
            if x.der is None:
                x.der = NodoBST(clave, valor); return raiz
            x = x.der
        else:
            x.valor = valor; return raiz


def altura_arbol(raiz):
    """Altura en aristas (iterativa). Árbol vacío = -1."""
    pila, h = [(raiz, 0)], -1
    while pila:
        nodo, d = pila.pop()
        if nodo is None:
            continue
        h = max(h, d)
        pila.append((nodo.izq, d + 1))
        pila.append((nodo.der, d + 1))
    return h


print("✅ Setup listo.")
print("🐍 Python", sys.version.split()[0])

# 🔵 BLOQUE 1 — Rotaciones e invariantes (50 minutos)

Todo el balanceo de un árbol rojo-negro cabe en tres operaciones $O(1)$. En este bloque
las implementas y compruebas que hacen exactamente lo que prometen.

## PARTE 1A: Trazar las rotaciones a mano (12 minutos)

Recuerda la convención: **el color se guarda en el nodo, pero describe el enlace que entra
a ese nodo desde su padre**.

Considera este subárbol, donde el enlace hacia `x` es **rojo** y el que entra a `h` es negro:

```
      h (negro)
     / \
    A   x  ← ROJO
       / \
      B   C
```

### Pregunta 1 — `rotar_izquierda(h)`
Dibuja el subárbol resultante e indica el color del enlace de entrada de **cada** nodo.

### Pregunta 2 — ¿se conserva el orden?
El recorrido in-order antes de rotar es `A h B x C`. ¿Cuál es después? ¿Qué te dice eso?

### Pregunta 3 — `cambiar_colores`
Si `h` tiene **ambos** hijos con enlace rojo, `cambiar_colores(h)` los pinta de negro y pinta
`h` de rojo. ¿Cambió el número de enlaces **negros** en el camino de la raíz a cualquier
hoja del subárbol? Justifica: esa es la razón por la que el invariante 3 se preserva.

> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: "¿Por qué una rotación nunca puede romper la
> propiedad de orden del árbol de búsqueda?"

### Tu Respuesta 1A (edita esta celda)

**P1 — resultado de `rotar_izquierda(h)`:**

```
(dibuja aquí)
```

**P2 — in-order después de rotar:**

**P3 — por qué `cambiar_colores` preserva el balance negro:**

## PARTE 1B: Implementar las tres operaciones (20 minutos)

> ⚠️ **Importante:** en `rotar_izquierda` y `rotar_derecha`, el nodo que sube **hereda** el
> color del que baja, y el que baja queda con enlace **rojo**. Si te saltas eso, el árbol
> sigue ordenado pero pierde el balance.

In [ ]:
def rotar_izquierda(h: "Nodo") -> "Nodo":
    """
    Endereza un enlace rojo que apunta a la derecha, dejándolo a la izquierda.

         h                    x
        / \                  / \
       A   x  ←ROJO  ==>    h   C
          / \        ROJO→ / \
         B   C            A   B

    Parámetros:
        h (Nodo): nodo cuyo hijo derecho tiene enlace rojo
    Retorna:
        Nodo: la nueva raíz del subárbol

    Complejidad:
        Temporal: O(1) — solo reasigna punteros
        Espacial: O(1)
    """
    # Tu código aquí.
    # 1. x = h.der
    # 2. h.der = x.izq   y   x.izq = h
    # 3. x hereda el color de h; h queda en ROJO
    # 4. actualiza los tamaños: x.n = h.n  y  h.n = 1 + tam(h.izq) + tam(h.der)
    # 5. retorna x
    pass


def rotar_derecha(h: "Nodo") -> "Nodo":
    """
    Inversa de rotar_izquierda: pasa un enlace rojo de la izquierda a la derecha.

    Parámetros:
        h (Nodo): nodo cuyo hijo izquierdo tiene enlace rojo
    Retorna:
        Nodo: la nueva raíz del subárbol

    Complejidad:
        Temporal: O(1)
        Espacial: O(1)
    """
    # Tu código aquí (simétrico al anterior).
    pass


def cambiar_colores(h: "Nodo") -> None:
    """
    Divide un 4-nodo temporal: ambos hijos pasan a NEGRO y h pasa a ROJO.

    Equivale a «la clave del medio sube al padre» en el árbol 2-3.

    Parámetros:
        h (Nodo): nodo con AMBOS hijos en rojo
    Retorna:
        None — modifica los colores in-place.

    Complejidad:
        Temporal: O(1)
        Espacial: O(1)
    """
    # Tu código aquí.
    pass

In [ ]:
def verificar_rotaciones(rot_izq, rot_der, cambiar):
    """Verifica las tres operaciones: forma, colores, orden in-order y tamaños."""

    def inorden(x, salida):
        if x is None:
            return
        inorden(x.izq, salida)
        salida.append(x.clave)
        inorden(x.der, salida)

    def construir_caso():
        """h(negro) con hijo derecho x(rojo); A, B, C hojas negras."""
        h = Nodo("h", 0, NEGRO); x = Nodo("x", 0, ROJO)
        A = Nodo("A", 0, NEGRO); B = Nodo("B", 0, NEGRO); Cc = Nodo("C", 0, NEGRO)
        h.izq, h.der = A, x
        x.izq, x.der = B, Cc
        h.n, x.n, A.n, B.n, Cc.n = 5, 3, 1, 1, 1
        return h

    aprobados, total = 0, 6

    # 1. rotar_izquierda: forma
    h = construir_caso()
    try:
        r = rot_izq(h)
        if r is not None and r.clave == "x" and r.izq.clave == "h":
            print("  ✅ rotar_izquierda: x sube y h queda a su izquierda"); aprobados += 1
        else:
            print(f"  ❌ rotar_izquierda: forma incorrecta (raíz={getattr(r,'clave',None)})")
    except Exception as e:
        print(f"  💥 rotar_izquierda — Error: {e}")

    # 2. rotar_izquierda: colores
    h = construir_caso()
    try:
        r = rot_izq(h)
        if r.color == NEGRO and r.izq.color == ROJO:
            print("  ✅ rotar_izquierda: x hereda el negro, h queda rojo"); aprobados += 1
        else:
            print("  ❌ rotar_izquierda: colores mal propagados "
                  f"(x={'ROJO' if r.color else 'NEGRO'}, h={'ROJO' if r.izq.color else 'NEGRO'})")
    except Exception as e:
        print(f"  💥 rotar_izquierda colores — Error: {e}")

    # 3. rotar_izquierda: preserva el in-order
    h = construir_caso()
    antes = []; inorden(h, antes)
    try:
        r = rot_izq(h)
        despues = []; inorden(r, despues)
        if antes == despues:
            print(f"  ✅ rotar_izquierda: preserva el orden {antes}"); aprobados += 1
        else:
            print(f"  ❌ rotar_izquierda: rompió el orden — antes {antes}, después {despues}")
    except Exception as e:
        print(f"  💥 rotar_izquierda orden — Error: {e}")

    # 4. rotar_izquierda: tamaños
    h = construir_caso()
    try:
        r = rot_izq(h)
        if r.n == 5 and r.izq.n == 3:
            print("  ✅ rotar_izquierda: tamaños actualizados"); aprobados += 1
        else:
            print(f"  ❌ rotar_izquierda: tamaños incorrectos (x.n={r.n}, h.n={r.izq.n}; se esperaba 5 y 3)")
    except Exception as e:
        print(f"  💥 rotar_izquierda tamaños — Error: {e}")

    # 5. rotar_derecha deshace rotar_izquierda
    h = construir_caso()
    antes = []; inorden(h, antes)
    try:
        r = rot_der(rot_izq(h))
        despues = []; inorden(r, despues)
        if r.clave == "h" and antes == despues:
            print("  ✅ rotar_derecha deshace rotar_izquierda"); aprobados += 1
        else:
            print(f"  ❌ rotar_derecha: no es la inversa (raíz={getattr(r,'clave',None)})")
    except Exception as e:
        print(f"  💥 rotar_derecha — Error: {e}")

    # 6. cambiar_colores
    p = Nodo("p", 0, NEGRO); a = Nodo("a", 0, ROJO); b = Nodo("b", 0, ROJO)
    p.izq, p.der = a, b
    try:
        cambiar(p)
        if p.color == ROJO and a.color == NEGRO and b.color == NEGRO:
            print("  ✅ cambiar_colores: hijos a negro, padre a rojo"); aprobados += 1
        else:
            print("  ❌ cambiar_colores: colores incorrectos")
    except Exception as e:
        print(f"  💥 cambiar_colores — Error: {e}")

    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados == total else f'⚠️  {aprobados}/{total} casos correctos'}")

verificar_rotaciones(rotar_izquierda, rotar_derecha, cambiar_colores)

## PARTE 1C: Armar el árbol completo (18 minutos)

Con las tres operaciones listas, `put` es la inserción de un BST **más tres líneas** al
volver de la recursión. Complétalas.

In [ ]:
class ArbolRojoNegro:
    """Symbol table ordenada sobre un árbol rojo-negro left-leaning."""

    def __init__(self):
        self.raiz = None

    def get(self, clave):
        """Busca el valor de una clave. Complejidad: O(log n) peor caso."""
        x = self.raiz
        while x is not None:
            if clave < x.clave:
                x = x.izq
            elif clave > x.clave:
                x = x.der
            else:
                return x.valor
        return None

    def __contains__(self, clave):
        return self.get(clave) is not None

    def __len__(self):
        return tam(self.raiz)

    def put(self, clave, valor):
        """Inserta o actualiza una clave. Complejidad: O(log n) peor caso."""
        self.raiz = self._put(self.raiz, clave, valor)
        self.raiz.color = NEGRO      # la raíz siempre es negra

    def _put(self, h, clave, valor):
        if h is None:
            return Nodo(clave, valor, ROJO)

        # ── inserción idéntica a la de un BST ──
        if clave < h.clave:
            h.izq = self._put(h.izq, clave, valor)
        elif clave > h.clave:
            h.der = self._put(h.der, clave, valor)
        else:
            h.valor = valor

        # ── LAS TRES LÍNEAS DEL BALANCEO — complétalas ──
        # 1. Si el hijo DERECHO es rojo y el IZQUIERDO no lo es -> rotar a la izquierda.
        # 2. Si el hijo IZQUIERDO es rojo y su hijo IZQUIERDO también -> rotar a la derecha.
        # 3. Si AMBOS hijos son rojos -> cambiar_colores.
        #
        # Tu código aquí (3 condicionales, en este orden):

        h.n = 1 + tam(h.izq) + tam(h.der)
        return h

    # ── diagnóstico (ya implementado) ─────────────────────────────────────
    def altura(self):
        return altura_arbol(self.raiz)

    def altura_negra(self):
        h, x = 0, self.raiz
        while x is not None:
            if not es_rojo(x):
                h += 1
            x = x.izq
        return h

    def es_valido(self):
        """Verifica los tres invariantes. Retorna (bool, mensaje)."""
        def chequear(x, negros, esperado):
            if x is None:
                return (negros == esperado), "balance negro roto"
            if es_rojo(x.der):
                return False, f"enlace rojo a la derecha en {x.clave!r}"
            if es_rojo(x) and es_rojo(x.izq):
                return False, f"dos enlaces rojos seguidos en {x.clave!r}"
            sig = negros + (0 if es_rojo(x) else 1)
            ok, msg = chequear(x.izq, sig, esperado)
            if not ok:
                return False, msg
            return chequear(x.der, sig, esperado)

        if self.raiz is None:
            return True, "árbol vacío"
        if es_rojo(self.raiz):
            return False, "la raíz debe ser negra"
        return chequear(self.raiz, 0, self.altura_negra())

    def claves_en_orden(self):
        """Recorrido in-order iterativo; debe devolver las claves ordenadas."""
        salida, pila, x = [], [], self.raiz
        while pila or x is not None:
            while x is not None:
                pila.append(x); x = x.izq
            x = pila.pop()
            salida.append(x.clave)
            x = x.der
        return salida


print("✅ Clase ArbolRojoNegro definida (completa el método _put).")

In [ ]:
def verificar_arbol(cls):
    """Verifica correctitud, invariantes y cota de altura del árbol."""
    casos = [
        (list(range(20)), "20 claves EN ORDEN (el peor caso del BST)"),
        (list(range(20, 0, -1)), "20 claves en orden INVERSO"),
        ([5, 3, 8, 1, 4, 7, 9, 2, 6], "inserción arbitraria"),
        ([1], "una sola clave"),
        ([7, 7, 7, 7], "clave repetida: debe actualizar, no duplicar"),
        (list(range(500)), "500 claves en orden"),
    ]
    aprobados = 0
    for claves, desc in casos:
        arbol = cls()
        try:
            for k in claves:
                arbol.put(k, f"v{k}")

            unicas = sorted(set(claves))
            ok_orden   = arbol.claves_en_orden() == unicas
            ok_tam     = len(arbol) == len(unicas)
            ok_valores = all(arbol.get(k) == f"v{k}" for k in unicas)
            ok_falta   = arbol.get("no_existe" if isinstance(claves[0], str) else -999) is None
            valido, msg = arbol.es_valido()
            cota = 2 * math.log2(len(unicas)) if unicas else 0
            ok_altura  = arbol.altura() <= cota

            if all([ok_orden, ok_tam, ok_valores, ok_falta, valido, ok_altura]):
                print(f"  ✅ {desc} — altura {arbol.altura()} ≤ {cota:.1f}")
                aprobados += 1
            else:
                print(f"  ❌ {desc}")
                if not valido:      print(f"     invariante roto: {msg}")
                if not ok_orden:    print("     el recorrido in-order no devuelve las claves ordenadas")
                if not ok_tam:      print(f"     tamaño {len(arbol)}, se esperaba {len(unicas)}")
                if not ok_valores:  print("     algún get() devolvió un valor incorrecto")
                if not ok_falta:    print("     get() de una clave ausente no devolvió None")
                if not ok_altura:   print(f"     altura {arbol.altura()} supera la cota {cota:.1f}")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados == len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_arbol(ArbolRojoNegro)

### ✋ Punto de control del Bloque 1

Muestra al ayudante:
1. Tu traza de la Parte 1A.
2. `verificar_rotaciones` y `verificar_arbol` en 🎉.
3. Explica en una frase por qué el orden de los tres condicionales de `_put` importa.

# 🟠 BLOQUE 2 — La garantía, medida (50 minutos)

Ya tienes un árbol que se balancea. Ahora comprueba experimentalmente que la garantía
teórica se cumple, y decide cuándo vale la pena pagarla.

## PARTE 2A: BST contra rojo-negro sobre las mismas claves (15 minutos)

Mide la altura de ambas estructuras alimentándolas con **exactamente la misma secuencia**
de claves, bajo tres órdenes distintos.

In [ ]:
def comparar_alturas(tamanos, orden):
    """
    Compara la altura del BST simple y del árbol rojo-negro.

    Parámetros:
        tamanos (list[int]): valores de n a probar
        orden (str): 'ordenado', 'inverso' o 'aleatorio'
    Retorna:
        (list[int], list[int], list[float]): alturas BST, alturas RN, cota 2·log2(n)
    """
    h_bst, h_rn, cota = [], [], []
    for n in tamanos:
        if orden == "ordenado":
            claves = list(range(n))
        elif orden == "inverso":
            claves = list(range(n - 1, -1, -1))
        else:
            claves = list(range(n)); random.shuffle(claves)

        r = None
        for k in claves:
            r = bst_insertar(r, k, k)
        h_bst.append(altura_arbol(r))

        a = ArbolRojoNegro()
        for k in claves:
            a.put(k, k)
        h_rn.append(a.altura())

        cota.append(2 * math.log2(n))
    return h_bst, h_rn, cota


tamanos = [50, 100, 200, 400, 800, 1600]
for orden in ["ordenado", "inverso", "aleatorio"]:
    hb, hr, ct = comparar_alturas(tamanos, orden)
    print(f"\nOrden de inserción: {orden.upper()}")
    print(f"{'n':>7} {'BST':>8} {'rojo-negro':>12} {'2·log₂n':>10}")
    print("-" * 40)
    for n, b, r, c in zip(tamanos, hb, hr, ct):
        print(f"{n:>7} {b:>8} {r:>12} {c:>10.1f}")

In [ ]:
# Gráfico de las tres situaciones
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, orden in zip(axes, ["ordenado", "inverso", "aleatorio"]):
    hb, hr, ct = comparar_alturas(tamanos, orden)
    ax.plot(tamanos, hb, "o-", linewidth=2, label="BST simple")
    ax.plot(tamanos, hr, "s-", linewidth=2, label="Rojo-negro")
    ax.plot(tamanos, ct, "--", linewidth=2, color="red", label="cota 2·log₂n")
    ax.set_title(f"Inserción {orden}")
    ax.set_xlabel("n")
    ax.set_yscale("log")
    ax.grid(alpha=0.3, which="both")
axes[0].set_ylabel("Altura (escala log)")
axes[0].legend()
plt.suptitle("El rojo-negro es plano en los tres escenarios; el BST solo sobrevive el aleatorio")
plt.tight_layout()
plt.show()

## PARTE 2B: ¿Cuánto cuesta el balanceo? (15 minutos)

La garantía no es gratis: cada `put` hace rotaciones y cambios de color. Mide el precio.

In [ ]:
# Tiempo de construcción y de búsqueda, BST vs rojo-negro vs dict nativo
N = 20000
claves_ale = list(range(N)); random.shuffle(claves_ale)
consultas = [random.randrange(N) for _ in range(20000)]

resultados = {}

# --- inserción ordenada ---
t0 = time.perf_counter()
a = ArbolRojoNegro()
for k in range(N):
    a.put(k, k)
t_rn_ord = time.perf_counter() - t0

t0 = time.perf_counter()
for k in consultas:
    a.get(k)
t_rn_ord_get = time.perf_counter() - t0

# --- inserción aleatoria ---
t0 = time.perf_counter()
a2 = ArbolRojoNegro()
for k in claves_ale:
    a2.put(k, k)
t_rn_ale = time.perf_counter() - t0

t0 = time.perf_counter()
for k in consultas:
    a2.get(k)
t_rn_ale_get = time.perf_counter() - t0

t0 = time.perf_counter()
r = None
for k in claves_ale:
    r = bst_insertar(r, k, k)
t_bst_ale = time.perf_counter() - t0

def bst_get(raiz, clave):
    x = raiz
    while x is not None:
        if clave < x.clave:   x = x.izq
        elif clave > x.clave: x = x.der
        else:                 return x.valor
    return None

t0 = time.perf_counter()
for k in consultas:
    bst_get(r, k)
t_bst_ale_get = time.perf_counter() - t0

# --- dict nativo ---
t0 = time.perf_counter()
d = {}
for k in claves_ale:
    d[k] = k
t_dict = time.perf_counter() - t0

t0 = time.perf_counter()
for k in consultas:
    d.get(k)
t_dict_get = time.perf_counter() - t0

print(f"n = {N} claves, {len(consultas)} consultas\n")
print(f"{'Estructura':<34}{'construir (s)':>15}{'consultar (s)':>15}")
print("-" * 64)
print(f"{'Rojo-negro (claves ORDENADAS)':<34}{t_rn_ord:>15.3f}{t_rn_ord_get:>15.3f}")
print(f"{'Rojo-negro (claves aleatorias)':<34}{t_rn_ale:>15.3f}{t_rn_ale_get:>15.3f}")
print(f"{'BST simple (claves aleatorias)':<34}{t_bst_ale:>15.3f}{t_bst_ale_get:>15.3f}")
print(f"{'dict nativo de Python (hash)':<34}{t_dict:>15.3f}{t_dict_get:>15.3f}")
print("\n👉 El BST con claves ALEATORIAS se CONSTRUYE más rápido, porque no paga")
print("   rotaciones. Pero CONSULTA más lento: su altura aleatoria (~1,39·log₂n)")
print("   es mayor que la del rojo-negro rebalanceado. Y esa ventaja de construcción")
print("   desaparece por completo si las claves llegan ordenadas.")
print("\n👉 El dict nativo gana por goleada... pero NO mantiene las claves ordenadas.")
print("   Ese es el tema de la próxima clase.")

## PARTE 2C: Lo que el hash no te puede dar (12 minutos)

Un `dict` es más rápido que cualquier árbol. Entonces, ¿para qué existen los árboles
balanceados? Porque el árbol mantiene el **orden**, y eso habilita consultas que una tabla
hash no puede responder sin recorrerlo todo.

Implementa dos de esas consultas sobre tu árbol.

In [ ]:
def rango(arbol, lo, hi):
    """
    Devuelve, EN ORDEN, todas las claves k del árbol con lo <= k <= hi.

    Es la consulta que una tabla hash no puede responder en menos de O(n).

    Parámetros:
        arbol (ArbolRojoNegro): el árbol
        lo, hi: extremos del rango, ambos inclusive
    Retorna:
        list: claves dentro del rango, en orden ascendente

    Complejidad esperada:
        O(log n + k), con k = número de claves reportadas
    """
    # Tu código aquí.
    # Pista: recorrido in-order que PODA — si la clave actual ya es > hi no hace
    # falta bajar por la derecha; si es < lo, no hace falta bajar por la izquierda.
    pass


def piso(arbol, clave):
    """
    Devuelve la mayor clave del árbol que sea <= clave, o None si no existe.

    Parámetros:
        arbol (ArbolRojoNegro): el árbol
        clave: valor de referencia
    Retorna:
        la clave encontrada, o None

    Complejidad esperada:
        O(log n)
    """
    # Tu código aquí.
    # Pista: baja desde la raíz llevando el mejor candidato visto hasta ahora.
    pass

In [ ]:
def verificar_consultas_de_orden(fn_rango, fn_piso):
    """Verifica rango() y piso() contra una implementación de referencia."""
    claves = [1, 3, 5, 7, 9, 11, 13, 15, 20, 25, 30]
    arbol = ArbolRojoNegro()
    for k in claves:
        arbol.put(k, k)
    ordenadas = sorted(claves)

    casos_rango = [
        ((5, 13),   [5, 7, 9, 11, 13], "rango interior"),
        ((0, 100),  ordenadas,          "rango que cubre todo"),
        ((4, 6),    [5],                "rango con una sola clave"),
        ((16, 19),  [],                 "rango vacío entre dos claves"),
        ((30, 30),  [30],               "rango de un punto, en el máximo"),
    ]
    casos_piso = [
        (10, 9,    "piso de un valor ausente"),
        (5,  5,    "piso de un valor presente"),
        (0,  None, "piso por debajo del mínimo"),
        (100, 30,  "piso por encima del máximo"),
        (19, 15,   "piso en un hueco"),
    ]

    aprobados = total = 0
    for (lo, hi), esperado, desc in casos_rango:
        total += 1
        try:
            r = fn_rango(arbol, lo, hi)
            r = list(r) if r is not None else None
            if r == esperado:
                print(f"  ✅ rango: {desc} → {r}"); aprobados += 1
            else:
                print(f"  ❌ rango: {desc}\n     Esperado: {esperado}\n     Obtenido: {r}")
        except Exception as e:
            print(f"  💥 rango: {desc} — Error: {e}")

    for k, esperado, desc in casos_piso:
        total += 1
        try:
            r = fn_piso(arbol, k)
            if r == esperado:
                print(f"  ✅ piso: {desc} → {r}"); aprobados += 1
            else:
                print(f"  ❌ piso: {desc}\n     Esperado: {esperado}\n     Obtenido: {r}")
        except Exception as e:
            print(f"  💥 piso: {desc} — Error: {e}")

    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados == total else f'⚠️  {aprobados}/{total} casos correctos'}")

verificar_consultas_de_orden(rango, piso)

## PARTE 2D: Cierre comparativo de la Unidad 4 (8 minutos)

Completa la tabla con lo que mediste hoy y en las sesiones anteriores.

| | Lista desordenada | Arreglo ordenado | BST simple | Rojo-negro | Tabla hash |
|---|---|---|---|---|---|
| `get` promedio | | | | | |
| `get` **peor caso** | | | | | |
| `put` promedio | | | | | |
| `put` **peor caso** | | | | | |
| ¿Mantiene el orden? | | | | | |
| ¿Soporta `rango` eficiente? | | | | | |

> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: "Un sistema de reservas necesita responder
> «¿qué reservas hay entre las 14:00 y las 16:00?». ¿Qué estructura eliges y por qué?"

### Preguntas de Análisis (edita esta celda)

1. En la Parte 2B, el BST simple con claves **aleatorias** resultó algo más rápido que el
   rojo-negro. ¿Por qué? ¿En qué escenario esa ventaja se convierte en un desastre?

2. El `dict` nativo fue el más rápido en las dos columnas. Da un caso de uso concreto
   —del tipo de sistema que te tocaría programar— en el que **igual** preferirías el árbol.

3. La cota es $h \le 2\log_2 n$. En tus mediciones, ¿qué tan cerca de esa cota llegó la
   altura real? ¿Es una cota ajustada o holgada?

4. ¿Qué operación del árbol 2-3 corresponde a `cambiar_colores`, y por qué esa
   correspondencia hace que el rebalanceo sea $O(1)$ en vez de propagarse hacia arriba
   explícitamente?

**Tus respuestas:**

## 🔬 Zona de Experimentación

Las siguientes celdas son tuyas. Algunas sugerencias:
- Comenta uno de los tres condicionales de `_put` y usa `es_valido()` para ver **qué**
  invariante se rompe primero y con cuántas claves.
- Busca la secuencia de inserción que maximice la altura del rojo-negro. ¿Qué tan cerca
  de $2\log_2 n$ consigues llegar?
- Implementa `select(k)`: la k-ésima clave más pequeña, usando el campo `n` de cada nodo.

In [ ]:
# Espacio libre para experimentar
# Sugerencia: rompe un invariante a propósito y observa el reporte de es_valido().


In [ ]:
# Espacio libre para experimentar
# Sugerencia: implementa select(k) aprovechando el campo n (tamaño del subárbol).


## 📤 Entrega de la Tarea de la Unidad 4

Al cierre de esta sesión se entrega la **Tarea de la Unidad 4** (producto computacional,
10% del Área N°4).

- Enunciado: [`S14_ALG_ASIG_BST.ipynb`](../S14_U4_BST/S14_ALG_ASIG_BST.ipynb)
- Formato: notebook ejecutado + informe breve, según indica el enunciado.

> ⚠️ **Prueba de la Unidad 4.** Cubre Symbol Tables, Binary Search Trees y árboles
> balanceados — todo lo trabajado hasta hoy. **Tablas Hash no entra**: se dicta después y
> se evalúa en el Examen Opcional Acumulativo.

## 📚 Lecturas Recomendadas y Práctica

### Textbooks

| Libro | Edición | Capítulo | Tema |
|-------|---------|----------|------|
| Cormen et al. (CLRS) — *Introduction to Algorithms* | 4ª ed. | Cap. 13.1–13.3 | Propiedades, rotaciones e inserción |
| Goodrich, Tamassia & Goldwasser (GTG) — *Data Structures and Algorithms in Python* | 1ª ed. | Cap. 11.2–11.5 | Árboles balanceados y sus variantes |
| Bhargava (Grok) — *Grokking Algorithms* | 2ª ed. | Cap. 8 | Intuición del rebalanceo |
| Miller & Ranum (M&R) — *Problem Solving with Algorithms and Data Structures Using Python* | 2011 | Cap. 6 | Árboles de búsqueda balanceados |

### Recursos gratuitos en línea

- 🌐 [Red/Black Tree Visualization (USFCA)](https://www.cs.usfca.edu/~galles/visualization/RedBlack.html) — inserta claves y observa cada rotación.
- 🌐 [VisuAlgo — BST / AVL](https://visualgo.net/en/bst) — comparación entre árbol balanceado y no balanceado.
- 📄 [Left-Leaning Red-Black Trees — Robert Sedgewick](https://sedgewick.io/wp-content/themes/sedgewick/papers/2008LLRB.pdf) — la variante implementada en este laboratorio.

### Práctica en Codeforces (soporta Python 3)

> 🔍 **Cómo filtrar:** ve a [codeforces.com/problemset](https://codeforces.com/problemset),
> escribe `data structures` o `sortings` en **Tags** y ajusta **Rating**.

| # | Problema | Rating | Por qué es útil |
|---|----------|--------|-----------------|
| 1 | [1213C — Book Reading](https://codeforces.com/problemset/problem/1213/C) | ⭐ 1200 | Consultas sobre un conjunto ordenado |
| 2 | [1005C — Summarize to the Power of Two](https://codeforces.com/problemset/problem/1005/C) | ⭐⭐ 1300 | Contrasta diccionario ordenado contra hash |
| 3 | [1234D — Distinct Characters Queries](https://codeforces.com/problemset/problem/1234/D) | ⭐⭐⭐ 1600 | Consultas de rango con actualizaciones |

⚠️ El problema 1 es el **mínimo esperado**. Los otros dos son desafío opcional.